# Minimal RAG with Evaluation

## 1. Setup

In [ ]:
!pip install datasets sentence-transformers faiss-cpu tqdm

## 2. Load Dataset

In [13]:
from datasets import load_dataset

docs = load_dataset('rag-datasets/rag-mini-wikipedia', 
                    'text-corpus',
                    split="passages")

texts = [d["passage"] for d in docs]

print(f'Loaded {len(texts)} documents')

Loaded 3200 documents


In [28]:
import pandas as pd
pd.DataFrame(docs)

,passage,id
0,"Uruguay (official full name in ; pron. , Eas...",0
1,"It is bordered by Brazil to the north, by Arge...",1
2,Montevideo was founded by the Spanish in the e...,2
3,The economy is largely based in agriculture (m...,3
4,"According to Transparency International, Urugu...",4
...,...,...
3195,"*In 2007, a duck in Tallahassee, Florida survi...",3196
3196,*A rare genetic mutation sees some ducks born ...,3197
3197,*The Moche people of ancient Peru worshipped n...,3198
3198,*Angel Wing - A disease common in ducks.,3199


In [29]:
# QA (for evaluation)
qa = load_dataset(
    "rag-datasets/rag-mini-wikipedia",
    "question-answer",
    split="test"
)

In [30]:
print(qa[1])
print(qa.column_names)

{'question': 'Did Lincoln sign the National Banking Act of 1863?', 'answer': 'yes', 'id': 2}
['question', 'answer', 'id']


## 3. Create Embeddings

In [31]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embed_model.encode(texts, 
                                    batch_size=32,
                                    show_progress_bar=True)

/home/amir/miniconda3/envs/llm/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/100 [00:00<?, ?it/s]

## 4. Build Vector Index (FAISS)

In [ ]:
import faiss
import numpy as np

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

index.add(np.array(doc_embeddings))

print('FAISS index built')

FAISS index built


In [ ]:
# faiss.IndexFlatIP

## 5. Retrieval Function

In [48]:
def retrieve(query, k=3):
    q_emb = embed_model.encode([query])
    D, I = index.search(q_emb, k)
    return [texts[i] for i in I[0]] #, I[0], D[0]

## 6. Prompt Builder

In [54]:
def build_prompt(query, contexts):
    context_block = '\n\n'.join([c for c in contexts])
    return f"""
            Answer the question using ONLY the context.

            Context:
            {context_block}

            Question:
            {query}
            Answer:
            """

## 7. Demo Query

In [55]:
context_block = '\n\n'.join([c for c in contexts])
context_block

"* Richard de Villamil. Newton, The man. G.D. Knox, London, 1931. Preface by Albert Einstein. Reprinted by Johnson Reprint Corporation, New York (1972).  \n\nTesla, also believed that much of Albert Einstein's relativity theory had already been proposed by RuÄ\x91er BoÅ¡koviÄ\x87, stating in an unpublished interview:\n\nIn a 2005 poll of the Royal Society of who had the greatest effect on the history of science, Newton was deemed more influential than Albert Einstein."

In [56]:
query = 'Who was Albert Einstein?'
contexts = retrieve(query, k=3)

for i, text in enumerate(contexts):
    print(f'\n--- Context {i+1} ---\n{text[:300]}...')

prompt = build_prompt(query, contexts)

print('\n--- Prompt sent to LLM ---\n')
print(prompt)


--- Context 1 ---
* Richard de Villamil. Newton, The man. G.D. Knox, London, 1931. Preface by Albert Einstein. Reprinted by Johnson Reprint Corporation, New York (1972).  ...

--- Context 2 ---
Tesla, also believed that much of Albert Einstein's relativity theory had already been proposed by RuÄer BoÅ¡koviÄ, stating in an unpublished interview:...

--- Context 3 ---
In a 2005 poll of the Royal Society of who had the greatest effect on the history of science, Newton was deemed more influential than Albert Einstein....

--- Prompt sent to LLM ---


            Answer the question using ONLY the context.

            Context:
            * Richard de Villamil. Newton, The man. G.D. Knox, London, 1931. Preface by Albert Einstein. Reprinted by Johnson Reprint Corporation, New York (1972).  

Tesla, also believed that much of Albert Einstein's relativity theory had already been proposed by RuÄer BoÅ¡koviÄ, stating in an unpublished interview:

In a 2005 poll of the Royal Society of who 

In [69]:
for sample in qa.select(range(5)):
    print(sample)

{'question': 'Was Abraham Lincoln the sixteenth President of the United States?', 'answer': 'yes', 'id': 0}
{'question': 'Did Lincoln sign the National Banking Act of 1863?', 'answer': 'yes', 'id': 2}
{'question': 'Did his mother die of pneumonia?', 'answer': 'no', 'id': 4}
{'question': "How many long was Lincoln's formal education?", 'answer': '18 months', 'id': 6}
{'question': 'When did Lincoln begin his political career?', 'answer': '1832', 'id': 8}


In [73]:
for sample in qa.select(range(5)):
    q = sample['question']
    a = sample['answer']
    context = retrieve(q, 3)

    print('\n===========================')
    print('Q:', q)
    print('A:', a)

    print('\nRetrieved contexts:')
    for i, c in enumerate(context):
        print(f'\n--- Context {i+1} ---\n{c[:300]}...')

    # hit = any(a.lower() in c.lower() for c in ctx)
    # print('Retrieval hit:', hit)


Q: Was Abraham Lincoln the sixteenth President of the United States?
A: yes

Retrieved contexts:

--- Context 1 ---
Young Abraham Lincoln...

--- Context 2 ---
Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion of slavery in the United States, "[I]n his short autobiography written for the 1860 presidential camp...

--- Context 3 ---
Sixteen months before his death, his son, John Quincy Adams, became the sixth President of the United States (1825 1829), the only son of a former President to hold the office until George W. Bush in 2001....

Q: Did Lincoln sign the National Banking Act of 1863?
A: yes

Retrieved contexts:

--- Context 1 ---
Lincoln believed in the Whig theory of the presidency, which left Congress to write the laws while he signed them, vetoing only those bills that threatened his war powers. Thus, he signed the Homestead Act

## 8. Evaluation Dataset

In [57]:
eval_queries = [
    ('Who was Alan Turing?', 'mathematician'),
    ('What is the capital of France?', 'Paris'),
    ('Who developed the theory of relativity?', 'Einstein'),
    ('What is photosynthesis?', 'process'),
]

## 9. Retrieval Evaluation (Recall@k)

In [58]:
def eval_recall_at_k(k=3):
    hits = 0
    for q, expected in eval_queries:
        contexts = retrieve(q, k=k)
        joined = ' '.join([c[0].lower() for c in contexts])
        if expected.lower() in joined:
            hits += 1
    return hits / len(eval_queries)

print('Recall@3:', eval_recall_at_k(3))

Recall@3: 0.0


## 10. Answer-Level Evaluation

In [59]:
def eval_answer_match():
    hits = 0
    for q, expected in eval_queries:
        contexts = retrieve(q, k=3)
        answer = build_prompt(q, contexts).lower()
        if expected.lower() in answer:
            hits += 1
    return hits / len(eval_queries)

print('Answer match (proxy):', eval_answer_match())

Answer match (proxy): 0.5


## 11. Semantic Evaluation (Optional)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_match(pred, target):
    emb1 = embed_model.encode([pred])
    emb2 = embed_model.encode([target])
    return cosine_similarity(emb1, emb2)[0][0]

print(semantic_match('Einstein was a physicist', 'Einstein'))